# Suite CONF — Conformance corpus

`conformance/python3/` holds each example architecture's store snapshot, as `mbse.Architecture.Systems.Conformance`
writes it. A change to what the package writes shows here first: rerun the writer, review the diff, and commit it.
Once a TypeScript implementation exists, its corpus will be compared with this one byte for byte.

In [ ]:
from pathlib import Path

from mbse.Schemas.Framework import Builders, JSON, Validators
from mbse.Architecture.Systems.Conformance import Corpus, write
from mbse.Architecture.Systems.Framework import Interactions

FOLDER = Path.cwd().parent.parent / "conformance" / "python3"

## CONF-01 · The committed snapshots are current

In [ ]:
assert sorted(p.stem for p in FOLDER.glob("*.json")) == sorted(Corpus.CASES)
for case in Corpus.CASES:
    assert (FOLDER / f"{case}.json").read_text() == write.snapshot(case), \
        f"{case}.json is stale: run uv run python -m mbse.Architecture.Systems.Conformance.write"

## CONF-02 · Each committed snapshot reads back, valid, and writes the same text

In [ ]:
for case in Corpus.CASES:
    text = (FOLDER / f"{case}.json").read_text()
    store = Builders.Store()
    JSON.FromJSON(store).Snapshot(text)
    Interactions.define(store)
    library = store.singleton("Interactions.Library")
    for entry in library.modules:
        assert Validators.Validate(store).Reachable(store.registered("Interactions.Module"), entry.module) == []
        assert Interactions.Check()(entry.module) == []
    assert JSON.ToJSON(store).Snapshot() + "\n" == text